# ProSIT 2 · V2-04 Final Chronological Holdout & Deployment Acceptance

**Purpose:** Evaluate the frozen V2 models exactly once on the chronological holdout; audit identity and model contracts; produce defensible diagnostics and deployment-ready evaluation records. **No model training or selection occurs here.**

Inputs (in project root): `data/processed_v2/`, `audits/v2/`, `experiments_v2/splits/`, `deployment/model_artifacts_v2_final/` (created locally by V2-03). This notebook deliberately does **not** use the old V1 `deployment/model_artifacts/` directory.

**Protocol:** The selected Random Forest + threshold and log-target HGB regressor are frozen from grouped development CV. Holdout performance is descriptive, not grounds to revise the models. Any source→engineered identity mismatches stop the analysis. Non-six-digit IDs are reported but permitted if mapping is consistent. Existing evaluation-marker files prevent accidental repeated testing.

In [ ]:
from pathlib import Path
import sys, json, warnings, hashlib, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from IPython.display import display
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, brier_score_loss, log_loss, precision_recall_curve, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score)

ROOT = Path.cwd()  # set to project root if notebook lives in a subfolder
PROCESSED=ROOT/'data'/'processed_v2'
AUDITS=ROOT/'audits'/'v2'
SPLITS=ROOT/'experiments_v2'/'splits'
ARTIFACTS=ROOT/'deployment'/'model_artifacts_v2_final'
OUTPUT=ROOT/'experiments_v2'/'final_holdout_v2'
TBL=OUTPUT/'tables';FIG=OUTPUT/'figures';PRED=OUTPUT/'predictions'
for p in (TBL,FIG,PRED): p.mkdir(parents=True, exist_ok=True)
# Evaluate holdout only after the artifacts and integrity audit are verified.
RUN_ONE_TIME_HOLDOUT=True
# If the results already exist, do not rerun. Delete marker intentionally only if a documented rerun is essential.
ALLOW_REEVALUATION=False
N_BOOT=400
RNG_SEED=42
print('Project:',ROOT.resolve())
print('Python',sys.version.split()[0], 'scikit-learn',sklearn.__version__)

## A. Source identity and feature-contract checks

The earlier V2 experiments raised ID-format warnings. Format alone is not grounds for rejection; *mapping mismatches are*. The source→engineered identity audit is required to be present and empty. This check only verifies that the existing audit says zero mismatches; for an independently auditable submission, keep the original `train.csv` and the V2 engineering audit together.

In [ ]:
def json_read(path):
    p=Path(path)
    if not p.is_file(): raise FileNotFoundError(p)
    return json.loads(p.read_text(encoding='utf-8'))

engineering=json_read(PROCESSED/'engineering_manifest_v2.json')
meta=json_read(ARTIFACTS/'model_metadata_v2.json')
features=json_read(PROCESSED/'F1_borrowing_primary.json')
assert meta.get('feature_set')=='F1_borrowing_primary', 'Wrong trained feature set'
assert features==meta.get('features'), 'Engineered and fitted feature orders differ'
assert len(features)==26, 'Unexpected F1 contract length'
assert meta.get('classification_model')=='random_forest', 'Classifier not frozen as selected'
assert meta.get('regression_model')=='hgb_log', 'Regressor not frozen as selected'
assert not meta.get('holdout_tested',False), 'Metadata indicates holdout was evaluated already. Audit before proceeding.'
assert engineering.get('input_sha256')==meta.get('source_sha256'), 'Training and engineering source hashes differ'
assert engineering.get('primary_uses_outcome_history') is False
assert not {'customer_id','ID','row_key','tbl_loan_id','target','Total_Amount','due_date','lender_id','country_id'} & set(features)
assert float(meta['classification_threshold'])>0 and float(meta['classification_threshold'])<1
for name in ('identity_mismatches.csv','invalid_customer_ids.csv'):
    path=AUDITS/name
    if not path.is_file(): raise FileNotFoundError(f'Required V2 audit missing: {path}')
    audit=pd.read_csv(path)
    print(name,':',len(audit),'flagged records')
    if name=='identity_mismatches.csv' and len(audit):
        raise RuntimeError('Source-to-engineered identity mismatches found; cannot validate loan histories. Rebuild V2.')
    if name=='invalid_customer_ids.csv' and len(audit):
        warnings.warn('Nonstandard customer ID formatting remains unresolved; record in model card.')
for filename in ('default_risk_pipeline_v2.joblib','historical_amount_pipeline_v2.joblib'):
    if not (ARTIFACTS/filename).is_file():raise FileNotFoundError(f'V2 model artifact absent: {ARTIFACTS/filename}')
if str(meta.get('sklearn_version')) != sklearn.__version__:
    raise RuntimeError(f'Saved with scikit-learn {meta.get("sklearn_version")}, runtime has {sklearn.__version__}. Use the matching environment.')
# Joblib loads pickle content. Only load files you generated and trust.
classifier=joblib.load(ARTIFACTS/'default_risk_pipeline_v2.joblib')
regressor=joblib.load(ARTIFACTS/'historical_amount_pipeline_v2.joblib')
assert list(classifier.classes_)==[0,1], 'Unexpected classification label order'
print('Validated frozen model contract:',meta['classification_model'],meta['regression_model'],'threshold',meta['classification_threshold'])

## B. Reconstruct the *saved* chronological partition

Use `row_key` to join the pre-existing assignment; verify `ID` and `customer_id` as well. Never create a new split or select a cutoff after inspecting the holdout.

In [ ]:
def read_partition(task):
    df=pd.read_csv(PROCESSED/f'{task}_dataset.csv',dtype={'ID':'string','customer_id':'string','tbl_loan_id':'string','loan_type':'string'})
    split=pd.read_csv(SPLITS/f'{task}_holdout_assignments.csv',dtype={'ID':'string','customer_id':'string'})
    assert df.row_key.is_unique and df.ID.is_unique
    assert split.row_key.is_unique and split.partition.isin(['development','final_holdout']).all()
    assert df.customer_id.notna().all() and df.customer_id.str.strip().ne('').all()
    assert df.disbursement_date.notna().all() and set(features)<=set(df.columns)
    split=split[['row_key','ID','customer_id','partition']]
    joined=df.merge(split,on='row_key',how='left',validate='one_to_one',suffixes=('','_audit'),indicator=True)
    assert joined._merge.eq('both').all() and len(joined)==len(df), 'Partition join failed'
    assert joined.ID.eq(joined.ID_audit).all() and joined.customer_id.eq(joined.customer_id_audit).all(), 'Identity differs from saved split'
    dev=joined.loc[joined.partition=='development'].copy()
    hold=joined.loc[joined.partition=='final_holdout'].copy()
    assert len(dev)>0 and len(hold)>0
    assert pd.to_datetime(dev.disbursement_date).max()<pd.to_datetime(hold.disbursement_date).min()
    for col in meta['numeric_features']:
        hold[col]=pd.to_numeric(hold[col],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for col in meta['categorical_features']:
        hold[col]=hold[col].astype('string')
    return dev,hold
cls_dev,cls_hold=read_partition('classification')
reg_dev,reg_hold=read_partition('regression')
assert cls_hold.target.isin([0,1]).all() and cls_hold.target.nunique()==2
assert pd.to_numeric(reg_hold.Total_Amount,errors='coerce').gt(0).all()
print('Classification dev/holdout:',len(cls_dev),len(cls_hold),'holdout defaults:',int(cls_hold.target.sum()))
print('Regression dev/holdout:',len(reg_dev),len(reg_hold))
assert 0<cls_hold.target.sum()<len(cls_hold), 'Holdout must have positive and negative labels'

## C. One-time frozen evaluation

This cell evaluates *exactly* the previously selected estimators and threshold, without fitting, tuning or modifying any parameters. A file marker is written after successful evaluation to protect against accidentally reusing this holdout for model selection.

In [ ]:
marker=OUTPUT/'HOLDOUT_ALREADY_EVALUATED.json'
if not RUN_ONE_TIME_HOLDOUT:
    raise RuntimeError('Set RUN_ONE_TIME_HOLDOUT=True only when ready for one-time final evaluation.')
if marker.exists() and not ALLOW_REEVALUATION:
    raise RuntimeError(f'Final holdout was already evaluated: {marker}. Review saved results instead of rerunning.')
Xc=cls_hold.loc[:,features]; Xr=reg_hold.loc[:,features]
yc=cls_hold.target.astype(int).to_numpy()
yr=pd.to_numeric(reg_hold.Total_Amount).to_numpy(dtype=float)
p=classifier.predict_proba(Xc)[:,list(classifier.classes_).index(1)]
r=np.asarray(regressor.predict(Xr),dtype=float)
assert np.isfinite(p).all() and np.all((p>=0)&(p<=1))
assert np.isfinite(r).all(), 'Regressor generated nonfinite values'
threshold=float(meta['classification_threshold'])
yhat=(p>=threshold).astype(int)
rclip=np.maximum(0,r)
assert len(yc)==len(p) and len(yr)==len(r)
cls_preds=cls_hold[['row_key','ID','customer_id','disbursement_date','loan_type']].copy()
cls_preds['actual']=yc;cls_preds['prob_default']=p;cls_preds['flag_above_threshold']=yhat
reg_preds=reg_hold[['row_key','ID','customer_id','disbursement_date','loan_type']].copy()
reg_preds['actual_amount_kes']=yr;reg_preds['predicted_amount_kes']=rclip;reg_preds['raw_prediction_kes']=r
cls_preds.to_csv(PRED/'classification_final_holdout_predictions.csv',index=False)
reg_preds.to_csv(PRED/'regression_final_holdout_predictions.csv',index=False)
result_cls={'task':'classification','model':meta['classification_model'],'n':len(yc),'n_defaults':int(yc.sum()),
    'default_prevalence':float(yc.mean()),'threshold':threshold,
    'pr_auc':average_precision_score(yc,p),'roc_auc':roc_auc_score(yc,p),
    'precision':precision_score(yc,yhat,zero_division=0),'recall':recall_score(yc,yhat,zero_division=0),
    'f1':f1_score(yc,yhat,zero_division=0),'brier':brier_score_loss(yc,p),
    'log_loss':log_loss(yc,np.clip(p,1e-12,1-1e-12))}
tn,fp,fn,tp=confusion_matrix(yc,yhat,labels=[0,1]).ravel()
result_cls.update({'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)})
result_reg={'task':'regression','model':meta['regression_model'],'n':len(yr),'mae_kes':mean_absolute_error(yr,rclip),
    'rmse_kes':float(np.sqrt(mean_squared_error(yr,rclip))),'r2':r2_score(yr,rclip),
    'log_mae':mean_absolute_error(np.log1p(yr),np.log1p(rclip)),
    'log_rmse':float(np.sqrt(mean_squared_error(np.log1p(yr),np.log1p(rclip)))),
    'negative_raw_predictions':int((r<0).sum())}
pd.DataFrame([result_cls]).to_csv(TBL/'classification_final_holdout_metrics.csv',index=False)
pd.DataFrame([result_reg]).to_csv(TBL/'regression_final_holdout_metrics.csv',index=False)
marker.write_text(json.dumps({'evaluated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'source_sha256':meta.get('source_sha256'),'feature_set':meta.get('feature_set'),
 'classification_model':meta['classification_model'],'regression_model':meta['regression_model'],
 'threshold':threshold,'classifier_artifact_sha256':hashlib.sha256((ARTIFACTS/'default_risk_pipeline_v2.joblib').read_bytes()).hexdigest(),
 'regressor_artifact_sha256':hashlib.sha256((ARTIFACTS/'historical_amount_pipeline_v2.joblib').read_bytes()).hexdigest()},indent=2),encoding='utf-8')
print('ONE-TIME FINAL HOLDOUT RESULTS');display(pd.DataFrame([result_cls]));display(pd.DataFrame([result_reg]))

## D. Uncertainty and diagnostic figures

Stratified bootstrap here resamples positives and negatives separately to avoid undefined PR-AUC draws. It estimates variability **conditional on observed class prevalence** and does not account for repeated customers. For a rigorous clustered interval, implement customer-level bootstrap and report it separately.

In [ ]:
rng=np.random.default_rng(RNG_SEED)
pos=np.flatnonzero(yc==1);neg=np.flatnonzero(yc==0)
cb=[];rb=[]
for b in range(N_BOOT):
    ix=np.r_[rng.choice(pos,len(pos),replace=True),rng.choice(neg,len(neg),replace=True)]
    yy=yc[ix]; pp=p[ix];pred=(pp>=threshold).astype(int)
    cb.append({'pr_auc':average_precision_score(yy,pp),'roc_auc':roc_auc_score(yy,pp),
               'f1':f1_score(yy,pred,zero_division=0),'precision':precision_score(yy,pred,zero_division=0),
               'recall':recall_score(yy,pred,zero_division=0)})
    ridx=rng.integers(0,len(yr),len(yr))
    rb.append({'mae_kes':mean_absolute_error(yr[ridx],rclip[ridx]),
               'rmse_kes':float(np.sqrt(mean_squared_error(yr[ridx],rclip[ridx])))})
def ci_table(values, point):
    a=pd.DataFrame(values)
    return pd.DataFrame([{'metric':k,'estimate':point[k], 'lower_95':a[k].quantile(.025), 'upper_95':a[k].quantile(.975)} for k in a.columns])
cc=ci_table(cb,result_cls);rr=ci_table(rb,result_reg)
cc.to_csv(TBL/'classification_bootstrap_95ci.csv',index=False)
rr.to_csv(TBL/'regression_bootstrap_95ci.csv',index=False)
display(cc);display(rr)
fig,axes=plt.subplots(1,3,figsize=(15,4))
precision,recall,_=precision_recall_curve(yc,p)
axes[0].plot(recall,precision,label=f'AP {result_cls["pr_auc"]:.3f}');axes[0].axhline(yc.mean(),color='gray',ls='--',label='Prevalence')
axes[0].set(xlabel='Recall',ylabel='Precision',title='Final holdout PR curve');axes[0].legend()
fpr,tpr,_=roc_curve(yc,p);axes[1].plot(fpr,tpr,label=f'AUC {result_cls["roc_auc"]:.3f}');axes[1].plot([0,1],[0,1],'--',color='gray');axes[1].set(xlabel='FPR',ylabel='TPR',title='Final holdout ROC');axes[1].legend()
ConfusionMatrixDisplay.from_predictions(yc,yhat,labels=[0,1],ax=axes[2],colorbar=False)
axes[2].set_title(f'Confusion matrix (t={threshold:.2f})')
fig.tight_layout();fig.savefig(FIG/'classification_final_diagnostics.png',dpi=180);plt.show()
# Quantile calibration; no calibration fitting on holdout.
cal=pd.DataFrame({'observed':yc,'p':p})
cal['bin']=pd.qcut(cal.p.rank(method='first'),q=min(10,len(cal)),labels=False,duplicates='drop')
agg=cal.groupby('bin').agg(n=('observed','size'),mean_predicted=('p','mean'),observed_rate=('observed','mean')).reset_index()
agg.to_csv(TBL/'classification_holdout_calibration_bins.csv',index=False)
fig,ax=plt.subplots(figsize=(6,4));ax.plot([0,1],[0,1],'--',color='gray')
ax.scatter(agg.mean_predicted,agg.observed_rate,s=np.maximum(20,agg.n/agg.n.max()*170))
ax.set(xlabel='Mean predicted default probability',ylabel='Observed default rate',title='Holdout calibration (quantile bins)')
fig.tight_layout();fig.savefig(FIG/'classification_holdout_calibration.png',dpi=180);plt.show()
errors=yr-rclip
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].scatter(np.log1p(yr),np.log1p(rclip),s=7,alpha=.3)
axes[0].set(xlabel='log1p actual KES',ylabel='log1p predicted KES',title='Predicted vs actual')
axes[1].scatter(np.log1p(rclip),errors,s=7,alpha=.3)
axes[1].axhline(0,color='gray',ls='--')
axes[1].set(xlabel='log1p predicted KES',ylabel='Actual minus predicted KES',title='Original-scale residuals')
fig.tight_layout();fig.savefig(FIG/'regression_final_diagnostics.png',dpi=180);plt.show()

## E. Subgroup checks, limitations, and model card

Subgroup results are descriptive, not fairness certification. Only report precision/recall when enough examples exist; small groups and unknown protected characteristics limit interpretation. This dataset cannot establish legal compliance or cross-market performance.

In [ ]:
groups=[]
for name,col in [('loan_type','loan_type'),('lender_id','lender_id'),('country_id','country_id')]:
    if col not in cls_hold.columns: continue
    for key,ix in cls_hold.groupby(col,dropna=False).groups.items():
        loc=cls_hold.index.get_indexer(ix)
        yy=yc[loc]; pp=p[loc]; dh=yhat[loc]
        base={'dimension':name,'group':str(key),'n':len(yy),'defaults':int(yy.sum()),'prevalence':float(yy.mean())}
        if len(yy)>=100 and yy.sum()>=5 and (yy==0).sum()>=5:
            base.update({'pr_auc':average_precision_score(yy,pp),'recall':recall_score(yy,dh,zero_division=0),
                         'precision':precision_score(yy,dh,zero_division=0),'flag_rate':float(dh.mean())})
        else: base.update({'pr_auc':np.nan,'recall':np.nan,'precision':np.nan,'flag_rate':np.nan})
        groups.append(base)
sub=pd.DataFrame(groups)
sub.to_csv(TBL/'classification_subgroup_audit.csv',index=False)
display(sub.sort_values(['dimension','n'],ascending=[True,False]).head(25))
size=pd.qcut(pd.Series(yr).rank(method='first'),4,labels=['Q1','Q2','Q3','Q4'])
rs=[]
for group in ['Q1','Q2','Q3','Q4']:
    sel=np.flatnonzero(size.astype(str).to_numpy()==group)
    rs.append({'amount_band':group,'n':len(sel),'mae_kes':mean_absolute_error(yr[sel],rclip[sel]),
               'rmse_kes':float(np.sqrt(mean_squared_error(yr[sel],rclip[sel])))})
rs=pd.DataFrame(rs);rs.to_csv(TBL/'regression_amount_band_audit.csv',index=False);display(rs)

card=f"""# V2 Model Card — final chronological holdout

- **Dataset:** engineered V2 from train.csv; SHA256 `{meta.get('source_sha256')}`.
- **Features:** F1 borrowing history ({len(features)} fields); no current target values or repayment outcomes.
- **Classifier:** {meta['classification_model']}; default = 1; threshold {threshold:.2f} frozen from grouped development OOF predictions.
- **Regressor:** {meta['regression_model']}; output denotes estimated *historical disbursement* in KES, not affordability or an authorised credit limit.
- **Validation:** five-fold grouped development CV for model selection; independent later-loan chronological holdout evaluated once.
- **Holdout default rate:** {yc.mean():.4%} ({yc.sum()} / {len(yc)}).
- **Classifier:** PR-AUC {result_cls['pr_auc']:.4f}, ROC-AUC {result_cls['roc_auc']:.4f}, precision {result_cls['precision']:.4f}, recall {result_cls['recall']:.4f}, F1 {result_cls['f1']:.4f}; TP {tp}, FP {fp}, TN {tn}, FN {fn}.
- **Regressor:** MAE {result_reg['mae_kes']:,.2f} KES, RMSE {result_reg['rmse_kes']:,.2f} KES, R² {result_reg['r2']:.4f}.
- **Known limitations:** earlier customer IDs failed six-digit formatting; only accept if identity mismatch audit is zero. Borrower-history availability is approximated by disbursement dates. Outcome maturity is proxied by due date (not actual known-label timestamp). Grouped CV can include past events of held-out borrowers in event-history features. Temporal shift, extremely imbalanced defaults, potential lender/proxy biases, and nonrepresentative geography remain risks.
- **Intended use:** educational credit-risk decision support and demonstration only, never sole automated lending or an actionable optimal loan amount.
- **Fairness:** exploratory subgroup analysis only; missing protected attributes and small groups preclude broad claims.
- **Probability calibration:** inspect holdout reliability plot; do not recalibrate on the sealed final holdout.
- **Environment:** exported using scikit-learn {meta.get('sklearn_version')}; loading joblib from untrusted sources is unsafe.
- **Governance:** human review, consent, privacy, access control, reason code validation and jurisdiction-specific legal review required before real deployment.
"""
(OUTPUT/'MODEL_CARD_V2.md').write_text(card,encoding='utf-8')
print('Saved model card and final holdout figures:',OUTPUT.resolve())

## F. Deployment handoff

**Do not retrain or re-export from this notebook.** The frozen `.joblib` pipeline files live at `deployment/model_artifacts_v2_final/`. Keep them with `model_metadata_v2.json` and the archived final holdout report. Copy the separate V2 Streamlit demo package into your project; it reads those files directly and refuses older V1 contracts. The demo accepts only already engineered F1 features, avoiding fabricated historical data.

When presenting, distinguish default-risk probability from risk flags and historical amount estimates from approved credit limits. A feature value is an input description, **not** a verified causal explanation or SHAP attribution.